In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
# Configuración de la conexión
server = '10.0.0.129'  # Reemplaza con el nombre o dirección IP de tu servidor SQL Server
database = 'DB_GENERAL'  # Reemplaza con el nombre de tu base de datos
username = 'sa'  # Reemplaza con tu nombre de usuario
password = 'Essalud23**'  # Reemplaza con tu contraseña
driver = 'ODBC Driver 17 for SQL Server'
# Cadena de conexión
connection_string = f'mssql+pyodbc://{username}:{password}@{server}/{database}?driver={driver}'

In [8]:
try:
    # Establecer la conexión con SQLAlchemy
    engine = create_engine(connection_string)

    # Especificar el esquema
    schema = 'jcardenas'

    # Construir la consulta SQL
    query = f'SELECT CODIGO_DEPENDENCIA, SIGLAS FROM jcardenas.H_DEPENDENCIA hd WHERE hd.ES_DEPGENERAL = 1'

    # Leer los resultados en un DataFrame de Pandas
    dependencias = pd.read_sql(query, engine)

except Exception as e:
    print(f'Error de conexión: {e}')

finally:
    # No es necesario cerrar la conexión con SQLAlchemy
    pass

In [9]:
dependencias

,CODIGO_DEPENDENCIA,SIGLAS
0,2,GCPP
1,19,GCGF
2,40,GCSPE
3,96,GCL
4,104,GCTIC
...,...,...
77,308,AGG-HAAV
78,309,AGG-SMVC
79,310,AGG-SCQA
80,314,OAG


In [17]:
import pandas as pd

# Suponiendo que tienes una conexión a tu base de datos llamada `engine`
# y un DataFrame llamado `dependencias` con las columnas CODIGO_DEPENDENCIA y SIGLAS

for index, row in dependencias.iterrows():
    codigo = row['CODIGO_DEPENDENCIA']
    siglas = row['SIGLAS']
    
    query = f"""SELECT
        t.ESTADO,
        t.CODIGO_TRABAJADOR,
        hd4.DEPENDENCIA AS DEPENDENCIA_PADRE3,
        hd3.DEPENDENCIA AS DEPENDENCIA_PADRE2,
        hd2.DEPENDENCIA AS DEPENDENCIA_PADRE1,
        hd.DEPENDENCIA,
        t.APELLIDOS_TRABAJADOR,
        t.NOMBRES_TRABAJADOR,
        t.CONDICION,
        t.EMAIL,
        CASE
            WHEN t.DIRECTOR = 0 THEN 'Trabajador'
            WHEN t.DIRECTOR = 1 THEN 'Director'
            WHEN t.DIRECTOR = 6 THEN 'Secretario'
            ELSE 'Otro'
        END AS Rol,
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM DB_NETLOGIN.core.MAE_USUARIO_ROL_INTRANET mur
                WHERE mur.CODIGO_TRABAJADOR = t.CODIGO_TRABAJADOR
                AND mur.ID_ROL IN (261, 345)
            ) THEN 1
            ELSE 0
        END AS FLAG_UTILITARIO
    FROM
        jcardenas.H_TRABAJADOR t
    LEFT OUTER JOIN
        jcardenas.H_DEPENDENCIA hd ON t.CODIGO_DEPENDENCIA = hd.CODIGO_DEPENDENCIA
    LEFT OUTER JOIN 
        jcardenas.H_DEPENDENCIA hd2 ON hd.CODDEP_RESPONSABLE = hd2.CODIGO_DEPENDENCIA
    LEFT OUTER JOIN
        jcardenas.H_DEPENDENCIA hd3 ON hd2.CODDEP_RESPONSABLE = hd3.CODIGO_DEPENDENCIA
    LEFT OUTER JOIN
        jcardenas.H_DEPENDENCIA hd4 ON hd3.CODDEP_RESPONSABLE = hd4.CODIGO_DEPENDENCIA
    WHERE hd.CODIGO_DEPENDENCIA = {codigo} OR hd2.CODIGO_DEPENDENCIA = {codigo} OR hd3.CODIGO_DEPENDENCIA = {codigo} OR hd4.CODIGO_DEPENDENCIA = {codigo}"""

    df = pd.read_sql(query, engine)
    df.to_csv(f'{siglas}.csv', index=False)


In [ ]:
import pandas as pd

# Suponiendo que tienes una conexión a tu base de datos llamada `engine`
# y un DataFrame llamado `dependencias` con las columnas CODIGO_DEPENDENCIA y SIGLAS

for index, row in dependencias.iterrows():
    codigo = row['CODIGO_DEPENDENCIA']
    siglas = row['SIGLAS']
    
    query = f"""SELECT
        t.ESTADO,
        t.CODIGO_TRABAJADOR,
        hd4.DEPENDENCIA AS DEPENDENCIA_PADRE3,
        hd3.DEPENDENCIA AS DEPENDENCIA_PADRE2,
        hd2.DEPENDENCIA AS DEPENDENCIA_PADRE1,
        hd.DEPENDENCIA,
        t.APELLIDOS_TRABAJADOR,
        t.NOMBRES_TRABAJADOR,
        t.CONDICION,
        t.EMAIL,
        CASE
            WHEN t.DIRECTOR = 0 THEN 'Trabajador'
            WHEN t.DIRECTOR = 1 THEN 'Director'
            WHEN t.DIRECTOR = 6 THEN 'Secretario'
            ELSE 'Otro'
        END AS Rol,
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM DB_NETLOGIN.core.MAE_USUARIO_ROL_INTRANET mur
                WHERE mur.CODIGO_TRABAJADOR = t.CODIGO_TRABAJADOR
                AND mur.ID_ROL IN (261, 345)
            ) THEN 1
            ELSE 0
        END AS FLAG_UTILITARIO
    FROM
        jcardenas.H_TRABAJADOR t
    LEFT OUTER JOIN
        jcardenas.H_DEPENDENCIA hd ON t.CODIGO_DEPENDENCIA = hd.CODIGO_DEPENDENCIA
    LEFT OUTER JOIN 
        jcardenas.H_DEPENDENCIA hd2 ON hd.CODDEP_RESPONSABLE = hd2.CODIGO_DEPENDENCIA
    LEFT OUTER JOIN
        jcardenas.H_DEPENDENCIA hd3 ON hd2.CODDEP_RESPONSABLE = hd3.CODIGO_DEPENDENCIA
    LEFT OUTER JOIN
        jcardenas.H_DEPENDENCIA hd4 ON hd3.CODDEP_RESPONSABLE = hd4.CODIGO_DEPENDENCIA
    WHERE hd.CODIGO_DEPENDENCIA = {codigo} OR hd2.CODIGO_DEPENDENCIA = {codigo} OR hd3.CODIGO_DEPENDENCIA = {codigo} OR hd4.CODIGO_DEPENDENCIA = {codigo}"""

    df = pd.read_sql(query, engine)
    df.to_excel(f'{siglas}.xlsx', index=False)  # Exporta a archivo Excel en lugar de CSV
